# Cost Controls: a per-project AI budget that blocks runaway spend

An agent stuck in a retry loop, or a prompt that drags a huge context into every turn, can spend a
month of model budget in an afternoon. **Cost Controls** give each Highflame project a daily and a
monthly budget in US dollars. Highflame prices every model call your agents make, adds it to the
project's spend, and blocks new prompts once the budget is used up.

One budget covers both ways an agent reaches a model:

| Path | How the spend reaches Highflame |
| --- | --- |
| **AI Gateway**: the agent's model calls go through Highflame | The gateway reports every call. Nothing to change in the agent. |
| **Custom Agents**: the agent calls its model directly and uses the Highflame SDK for guardrails | The agent reports each call's token counts with one SDK call. |

This notebook spends real money on a tiny budget and watches it stop:

1. Read the project's budget, without spending anything.
2. Make a model call through the AI Gateway.
3. Make a model call from a custom agent that reports its usage.
4. Keep going until the budget is used up, and see both paths blocked.
5. See where the money went, by agent and by model.

## How it works

1. The agent calls a model.
2. The call's token counts reach Highflame: the gateway sends them itself, and a custom agent sends
   them with `usage.report()`.
3. Highflame prices the call from one price table, the same for every gateway and every SDK, and
   adds it to the project's spend for the day and for the month (UTC).
4. On the **next** prompt, the Cost Controls policy compares that spend with the budget, and blocks
   once it is used up.

The price of a call is known only after the model answers, so the call that crosses the budget
completes and the one after it is blocked. If that overshoot matters, use the **Monthly Budget
Headroom** template, which blocks at 95% of the budget.

## Before you start: set it up once in Studio

Use a project that nothing else depends on. When the budget is used up, **every** prompt in the
project is blocked until the next UTC day, or until you raise the budget.

1. Sign in at [studio.highflame.ai](https://studio.highflame.ai) and create a project, for example
   `cost-controls-demo`.
2. In that project, create an API key: **AI Gateway → Settings → API Keys**.
3. **Custom Agents → Policies → Cost Controls → Set budget.** Set **Daily budget (USD)** to `0.05`,
   leave the monthly budget blank, and select **Deploy**. This one budget applies to Custom Agents
   and to the AI Gateway.
4. In the same tab, deploy the **Daily Budget** template. The deploy dialog starts on **Monitor**,
   which only records; set **Enforcement Mode** to **Enforce**.
5. **AI Gateway → Policies → Cost Controls**: deploy **Daily Budget** the same way, in **Enforce**.

With `gpt-4o` each call below costs about one cent, so a $0.05 budget runs out in a handful of
calls. That is the point: the notebook should hit the limit in front of you.

## Setup

Copy `.env.example` to `.env` beside this notebook.

| Variable | What it is |
| --- | --- |
| `HIGHFLAME_API_KEY` | **Required.** The API key from step 2, in the project from step 1. |
| `PROVIDER_API_KEY` | **Required.** Your OpenAI API key. The gateway forwards it upstream, and the custom agent calls OpenAI with it directly. `OPENAI_API_KEY` works as a fallback. |
| `HIGHFLAME_GATEWAY_BASE_URL` | The LLM Base URL from **AI Gateway → LLM Providers**, with `/v1` appended. Defaults to `https://gateway.highflame.ai/llm/v1`. |
| `MODEL` | An OpenAI model. Defaults to `gpt-4o`, which costs enough per call to reach a small budget quickly. |
| `DAILY_BUDGET_USD` | The daily budget you set in step 3, used only to print dollar amounts. Defaults to `0.05`. |

In [1]:
%pip install -q -r requirements.txt

from importlib.metadata import version

print("highflame", version("highflame"))

Note: you may need to restart the kernel to use updated packages.
highflame 0.10.0


In [2]:
import json
import os
import time
import urllib.parse
import urllib.request
import uuid
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv
from highflame import Highflame
from highflame.zeroid import ToolScope
from openai import OpenAI

load_dotenv()

HIGHFLAME_API_KEY = os.environ["HIGHFLAME_API_KEY"]
PROVIDER_API_KEY = os.environ.get("PROVIDER_API_KEY") or os.environ.get("OPENAI_API_KEY", "")
if not PROVIDER_API_KEY:
    raise RuntimeError("Set PROVIDER_API_KEY (or OPENAI_API_KEY) in .env to your OpenAI API key.")
GATEWAY_BASE_URL = os.environ.get(
    "HIGHFLAME_GATEWAY_BASE_URL", "https://gateway.highflame.ai/llm/v1"
).rstrip("/")
MODEL = os.environ.get("MODEL", "gpt-4o")
DAILY_BUDGET_USD = float(os.environ.get("DAILY_BUDGET_USD", "0.05"))

# The platform API and the identity service. Leave both unset for the hosted product.
API_URL = os.environ.get("HIGHFLAME_API_URL", "https://api.highflame.ai").rstrip("/")
AUTH_URL = os.environ.get("HIGHFLAME_AUTH_URL", "https://auth.highflame.ai").rstrip("/")
SDK_URLS = {"base_url": API_URL, "identity_base_url": AUTH_URL, "token_url": f"{AUTH_URL}/oauth2/token"}

RUN_ID = uuid.uuid4().hex[:6]
CREATED: list[tuple[str, str]] = []  # (label, identity id) for the clean-up cell

highflame = Highflame(api_key=HIGHFLAME_API_KEY, **SDK_URLS)
print("account:", highflame.whoami()["account_id"])
print("model  :", MODEL, " budget:", f"${DAILY_BUDGET_USD:.2f}/day")

account: 757038846364
model  : gpt-4o  budget: $0.05/day


### Two agents, each with its own key

Spend is counted per project, and also per agent when the caller is a registered agent. Register
one agent for each path, so the report at the end can say which one spent what.

In [3]:
def register_agent(name: str, slug: str):
    agent = highflame.agents.register(
        name=name,
        external_id=f"cost-{slug}-{RUN_ID}",
        identity_type="agent",
        sub_type="tool_agent",
        trust_level="first_party",
        framework="openai",
        description="Cost Controls notebook demo. Safe to delete.",
        allowed_scopes=[ToolScope.READ, ToolScope.EXECUTE],
    )
    CREATED.append((name, agent.agent.id))
    return agent


gateway_agent = register_agent("Gateway Agent", "gateway")
custom_agent = register_agent("Custom Agent", "custom")

# The custom agent's own Highflame client: its guard checks and usage reports name the agent.
custom_hf = Highflame(api_key=custom_agent.api_key, **SDK_URLS)
print("registered:", gateway_agent.agent.name, "and", custom_agent.agent.name)

registered: Gateway Agent and Custom Agent


## 1. Read the budget, without spending anything

`budget_state()` asks Highflame how much of the project's budget is used, the same way a policy
sees it. It runs a guard check in `monitor` mode with `debug=True`, which returns the values the
Cost Controls policies read:

| Value | Meaning |
| --- | --- |
| `budget_cost_pct_daily` | Share of today's budget used, 0 to 100 |
| `budget_exceeded_daily` | `True` once today's budget is used up |
| `agent_spend_cents_daily` | What the calling agent spent today, in US cents |

They appear only when they apply: the budget values only when the project has a daily budget, and
the agent value only when the caller is a registered agent.

In [4]:
BUDGET_KEYS = (
    "budget_cost_pct_daily",
    "budget_exceeded_daily",
    "budget_cost_pct_monthly",
    "budget_exceeded_monthly",
    "agent_spend_cents_daily",
    "agent_spend_cents_monthly",
)


def budget_state(client: Highflame = highflame) -> dict:
    # mode="monitor": a read, never a block. The check costs nothing; it carries no tokens.
    decision = client.guard.evaluate_prompt("budget check", mode="monitor", debug=True)
    seen = {**(decision.context or {}), **(decision.projected_context or {})}
    return {key: seen[key] for key in BUDGET_KEYS if key in seen}


def meter(state: dict) -> str:
    pct = int(state.get("budget_cost_pct_daily", 0))
    bar = "#" * (pct // 5) + "." * (20 - pct // 5)
    used = DAILY_BUDGET_USD * pct / 100
    flag = "  USED UP" if state.get("budget_exceeded_daily") else ""
    return f"[{bar}] {pct:3d}% of ${DAILY_BUDGET_USD:.2f} (~${used:.3f}){flag}"


state = budget_state()
if "budget_cost_pct_daily" not in state:
    raise RuntimeError(
        "This project has no daily budget. Do step 3 of 'Before you start' in Studio, then run this cell again."
    )
print("project today:", meter(state))
if state.get("budget_exceeded_daily"):
    print(
        "\nToday's budget is already used up, probably by an earlier run. Raise the budget in Studio"
        " (Cost Controls -> Edit budget), or run this again after 00:00 UTC."
    )

project today: [....................]   0% of $0.05 (~$0.000)


## 2. A model call through the AI Gateway

The gateway speaks the OpenAI API, so the ordinary OpenAI client works. Your OpenAI key goes in
`Authorization` and is forwarded upstream; the agent's Highflame key goes in `X-Highflame-APIKey`.
There is no usage code at all: the gateway reports every call it forwards.

When the gateway blocks a prompt it still answers with a normal completion, so an existing client
keeps working. The completion id then starts with `chatcmpl-blocked`.

Every prompt below starts with a unique line. That keeps the provider's prompt cache out of the
way, so each call costs about the same.

In [5]:
# A support backlog to summarise: about 3,000 tokens of input, so each gpt-4o call costs about a cent.
ISSUES = ["login fails", "invoice is wrong", "export times out", "app crashes on start", "password reset email never arrives"]
PRODUCTS = ["the mobile app", "the web portal", "the billing page", "the admin console"]
BACKLOG = "\n".join(
    f"Ticket {n}: a customer reports that {ISSUES[n % 5]} in {PRODUCTS[n % 4]} since the last release."
    for n in range(1, 151)
)
TASK = "Write a short paragraph on each of the five most common problems in these tickets, with how many tickets mention it."


def work_prompt(step: int) -> str:
    return f"Request {step} of run {RUN_ID}.\n\n{TASK}\n\n{BACKLOG}"


gateway = OpenAI(
    base_url=GATEWAY_BASE_URL,
    api_key=PROVIDER_API_KEY,  # -> Authorization, forwarded upstream to OpenAI
    default_headers={"X-Highflame-APIKey": gateway_agent.api_key},
)


def gateway_call(step: int) -> dict:
    reply = gateway.chat.completions.create(
        model=f"openai/{MODEL}",  # the gateway routes on provider/model
        messages=[{"role": "user", "content": work_prompt(step)}],
        max_tokens=400,
    )
    allowed = not reply.id.startswith("chatcmpl-blocked")
    usage = reply.usage
    return {
        "path": "AI Gateway",
        "allowed": allowed,
        "tokens": f"{usage.prompt_tokens}/{usage.completion_tokens}" if (allowed and usage) else "-",
        "cost": "reported by the gateway" if allowed else "-",
        "text": reply.choices[0].message.content or "",
    }


result = gateway_call(1)
print("allowed:", result["allowed"], " tokens in/out:", result["tokens"])
print("reply  :", result["text"][:200].replace("\n", " "), "...")
time.sleep(2)  # the gateway reports usage just after it answers
print("\nproject today:", meter(budget_state()))

allowed: True  tokens in/out: 3099/400
reply  : Based on the 150 tickets provided, the five most common issues are identified as follows:  1. **Invoice Errors** (25 tickets): Many customers experienced incorrect invoices appearing across various pl ...



project today: [####................]  23% of $0.05 (~$0.012)


## 3. A model call from a custom agent

A custom agent calls OpenAI directly and uses the Highflame SDK for guardrails. Two SDK calls
matter here:

- `guard.evaluate_prompt()` **before** the model call. This is where a used-up budget blocks.
- `usage.report()` **after** the model call, with the token counts from the provider's response.
  This is the only addition the budget needs. Highflame prices the call and returns the price.

Report each call once. OpenAI counts cached prompt tokens inside `prompt_tokens`, so the agent sends
them separately as `cache_read_tokens`, where they are priced at the cached rate.

In [6]:
openai_direct = OpenAI(api_key=PROVIDER_API_KEY)  # straight to OpenAI, no gateway
SESSION = f"cost-demo-{RUN_ID}"


def custom_agent_call(step: int) -> dict:
    prompt = work_prompt(step)

    # 1. Guard the prompt. A used-up budget is refused here, before any money is spent.
    decision = custom_hf.guard.evaluate_prompt(prompt, session_id=SESSION)
    if not decision.allowed:
        return {"path": "Custom Agent", "allowed": False, "tokens": "-", "cost": "-", "text": decision.policy_reason or ""}

    # 2. Call the model directly.
    reply = openai_direct.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}], max_tokens=400
    )
    usage = reply.usage
    cached = getattr(usage.prompt_tokens_details, "cached_tokens", 0) or 0

    # 3. Report what the call used. Highflame prices it and adds it to the project's spend.
    report = custom_hf.usage.report(
        model=MODEL,
        provider="openai",
        input_tokens=usage.prompt_tokens - cached,
        cache_read_tokens=cached,
        output_tokens=usage.completion_tokens,
        session_id=SESSION,
    )
    return {
        "path": "Custom Agent",
        "allowed": True,
        "tokens": f"{usage.prompt_tokens}/{usage.completion_tokens}",
        "cost": f"${report.cost_micros / 1e6:.4f}" + ("" if report.priced else " (model not priced)"),
        "text": reply.choices[0].message.content or "",
    }


result = custom_agent_call(2)
print("allowed:", result["allowed"], " tokens in/out:", result["tokens"], " priced by Highflame:", result["cost"])
print("reply  :", result["text"][:200].replace("\n", " "), "...")
print("\nproject today:", meter(budget_state()))

allowed: True  tokens in/out: 3099/362  priced by Highflame: $0.0114
reply  : Based on the analysis of the ticket data, the five most common problems reported are as follows:  1. **Invoice Errors (28 Tickets):** A recurring issue for many users is the incorrect display of invoi ...



project today: [#########...........]  46% of $0.05 (~$0.023)


## 4. Keep going until the budget is used up

The two agents take turns. Both count against the same project budget, so the bar keeps climbing
whichever path made the call. Once it reaches 100%, the next prompt on **either** path is blocked.

In [7]:
rows = []
blocked = {"AI Gateway": None, "Custom Agent": None}
for step in range(3, 21):
    call = gateway_call if step % 2 else custom_agent_call
    result = call(step)
    time.sleep(2)  # let the gateway's report land before the next read
    state = budget_state()
    rows.append((step, result, meter(state)))
    print(
        f"{step:>2}  {result['path']:<12}  {'allowed' if result['allowed'] else 'BLOCKED':<8}"
        f"  tokens {result['tokens']:<9}  {result['cost']:<24}  {meter(state)}"
    )
    if not result["allowed"] and blocked[result["path"]] is None:
        blocked[result["path"]] = result["text"]
    if all(blocked.values()):
        break
    if state.get("budget_exceeded_daily") and result["allowed"] and len(rows) > 4 and rows[-2][1]["allowed"]:
        print(
            "\nThe budget is used up but calls are still allowed. Check that the Daily Budget template is"
            " deployed in Enforce mode under both Custom Agents and AI Gateway -> Policies -> Cost Controls."
        )
        break

 3  AI Gateway    allowed   tokens 3099/393   reported by the gateway   [#############.......]  69% of $0.05 (~$0.035)


 4  Custom Agent  allowed   tokens 3099/355   $0.0113                   [##################..]  92% of $0.05 (~$0.046)


 5  AI Gateway    allowed   tokens 3099/400   reported by the gateway   [####################] 100% of $0.05 (~$0.050)  USED UP


 6  Custom Agent  BLOCKED   tokens -          -                         [####################] 100% of $0.05 (~$0.050)  USED UP


 7  AI Gateway    BLOCKED   tokens -          -                         [####################] 100% of $0.05 (~$0.050)  USED UP


In [8]:
print("What the AI Gateway answered:\n ", (blocked["AI Gateway"] or "(not blocked)")[:300])
print("\nWhy the custom agent's prompt was refused:\n ", (blocked["Custom Agent"] or "(not blocked)")[:300])

What the AI Gateway answered:
  Highflame Security: Prompt blocked: this project has used its daily AI budget.

Why the custom agent's prompt was refused:
  Enterprise Policies Triggered: Daily Budget


Both paths stopped on one budget. The custom agent was refused **before** its model call, so the
refused prompts cost nothing. The gateway's refusal is a normal completion, so the calling code did
not break.

## 5. Where the money went

Highflame keeps one record per priced call, and serves it as a chargeback report grouped by agent,
by user, or by model. Studio shows the same report under **Observatory → Chargeback**.

The report is built from a stream, so the newest calls can take a minute or two to appear. The cell
waits for them.

In [9]:
form = urllib.parse.urlencode({"grant_type": "api_key", "api_key": HIGHFLAME_API_KEY}).encode()
with urllib.request.urlopen(
    urllib.request.Request(f"{AUTH_URL}/oauth2/token", data=form, method="POST",
                           headers={"Content-Type": "application/x-www-form-urlencoded"}),
    timeout=30,
) as response:
    read_token = json.loads(response.read())["access_token"]


def chargeback(group_by: str) -> dict:
    now = datetime.now(timezone.utc)
    query = urllib.parse.urlencode({
        "start": now.strftime("%Y-%m-%dT00:00:00Z"),
        "end": (now + timedelta(minutes=1)).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "group_by": group_by,
    })
    request = urllib.request.Request(
        f"{API_URL}/v1/obs/costs/chargeback?{query}", headers={"Authorization": f"Bearer {read_token}"}
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        return json.loads(response.read())


ours = {gateway_agent.agent.external_id, custom_agent.agent.external_id}
for _ in range(24):  # up to about 4 minutes
    by_agent = chargeback("agent")
    if ours <= {row.get("key") for row in by_agent.get("rows", [])}:
        break
    time.sleep(10)

def show(title: str, report: dict) -> None:
    print(title)
    for row in report.get("rows", []):
        if row["cost_usd"] > 0:  # calls at $0 (an unpriced model, for one) carry no cost to explain
            label = row["key"] or "(not a registered agent)"
            print(f"  {label:<28} ${row['cost_usd']:.4f}  {row['calls']:>3} calls  {row['share']:.0%}")


print(f"Today, this project: ${by_agent.get('total_cost_usd', 0):.4f} over {by_agent.get('total_calls', 0)} calls\n")
show("By agent", by_agent)
show("\nBy model", chargeback("model"))

Today, this project: $0.0578 over 5 calls

By agent
  cost-gateway-638ffe          $0.0352    3 calls  61%
  cost-custom-638ffe           $0.0227    2 calls  39%



By model
  openai/gpt-4o                $0.0578    5 calls  100%


## 6. Per-agent budgets

The project budget caps the total. You can also cap one agent: Highflame counts each registered
agent's spend, and the **Agent Monthly Budget** template blocks an agent once it has spent $200 in a
month. Attach it to the agent and edit the threshold for that agent. Its spend still counts against
the project's budget too.

The cell below reads what each agent spent today, as that template sees it.

In [10]:
gateway_hf = Highflame(api_key=gateway_agent.api_key, **SDK_URLS)
for name, client in (("Gateway Agent", gateway_hf), ("Custom Agent", custom_hf)):
    cents = budget_state(client).get("agent_spend_cents_daily")
    print(f"{name:<14} spent today: {cents if cents is not None else '-'} cents")

Gateway Agent  spent today: 3 cents


Custom Agent   spent today: 2 cents


## Clean up, and run it again

The agents are deactivated rather than erased, so their names stay taken; that is why each name
carries the run id.

The project stays blocked for the rest of the UTC day. To run the notebook again today, raise the
budget in Studio (**Cost Controls → Edit budget**). To turn budgets off, remove the budget: with no
budget, the Cost Controls templates never fire.

In [11]:
for label, identity_id in reversed(CREATED):
    try:
        highflame.agents.delete(identity_id)
        print("deleted:", label)
    except Exception as exc:
        print(f"clean-up skipped for {label}: {str(exc)[:60]}")

deleted: Custom Agent


deleted: Gateway Agent


## What this showed

- One daily budget in US dollars covered model calls through the AI Gateway and model calls a custom
  agent made directly.
- Highflame priced every call from one price table, and blocked the next prompt on both paths once
  the budget was used up. The custom agent was refused before it spent anything more.
- The gateway needed no code. The custom agent needed one `usage.report()` call after each model
  call.
- The chargeback report named the agent and the model behind every cent.

**Good to know**

- Spend is known only after a model answers, so the call that crosses the budget completes. The
  Monthly Budget Headroom template blocks at 95% to keep the overshoot small.
- A model that is not in Highflame's price table is counted at $0.
- Days and months are UTC.
- ChatGPT and Claude subscription traffic is paid for by the subscription, so it is not counted.